# Sudoku-05-PSO : Optimisation par Essaim Particulaire (Rust / evcxr)

**Navigation** : [<< Sudoku-04 SimulatedAnnealing Python](Sudoku-04-SimulatedAnnealing-Python.ipynb) | [Index](README.md) | [Sudoku-06 AIMA-CSP Python >>](Sudoku-06-AIMA-CSP-Python.ipynb)

Portage Rust du carnet [Sudoku-05-PSO-Python](Sudoku-05-PSO-Python.ipynb) (jumeau C# : [Sudoku-05-PSO-CSharp](Sudoku-05-PSO-CSharp.ipynb)), execute par le kernel **evcxr**. Le moteur PSO est ecrit en Rust standard ; la seule dépendance externe sert la demonstration sur les générateurs aléatoires.

## Objectifs d'apprentissage

A la fin de ce carnet, vous saurez :

1. Écrire en Rust les pieces d'un essaim particulaire applique au Sudoku : grille, opérateurs par bloc, fonction d'erreur, organismes workers et explorers, boucle d'evolution.
2. Reconnaitre **l'ecueil de traduction** propre aux métaheuristiques : en Python la graine est un argument (`random.Random(attempt)`), en Rust le générateur est une **valeur typee** qui circule explicitement — et le générateur global du fil, commode, n'est pas reproductible.
3. Porter le générateur MT19937 de CPython pour que, sur la **même graine**, le Rust rende exactement la trajectoire et la solution du jumeau Python.
4. Mesurer l'écart de performance entre les deux implementations sur le même algorithme et la même instance.

Le carnet suit la même progression que le jumeau Python : grille, opérateurs, erreur, solveur, tests, exercices. Chaque cellule de code est suivie de sa lecture, et les valeurs-pivots du jumeau Python sont reproduites a l'identique.

## 1. L'ecueil de traduction : la graine est une valeur, pas un global

En Python, l'aléatoire se prend dans un objet global par defaut, et une graine se pose en une ligne :

```python
rng = random.Random(attempt)   # generateur local, reproductible
rng.random()
```

En Rust idiomatique, deux routes existent, et elles ne se valent pas :

| Route | Forme | Reproductible ? |
|---|---|---|
| Générateur global du fil | `rand::rng()` | **non** — puise dans l'entropie du système |
| Générateur seme, passe explicitement | `StdRng::seed_from_u64(42)` | **oui**, mais suite propre a `StdRng` |

La seconde route est la bonne pratique Rust : le générateur est une **valeur** que l'on possede et que l'on passe en `&mut`. C'est aussi ce qui rend le portage comparable : une fonction qui recoit `rng: &mut Mt19937` ne peut pas tirer un nombre « en cachette ».

Reste une question que le domaine impose : quelle suite reproduire ? `StdRng` est un générateur différent de celui de CPython — mêmes graines, suites différentes, donc trajectoires différentes. Pour que le portage Rust soit **verifiable contre le jumeau Python** (même graine, même solution), ce carnet porte le générateur de CPython lui-même (section 2). La cellule suivante mesure d'abord l'écart entre les deux routes Rust.

In [2]:
:dep rand = "0.9"
use rand::rngs::StdRng;
use rand::{Rng, SeedableRng};

// Deux generateurs semes pareil rendent la meme suite : la graine suffit.
let mut a = StdRng::seed_from_u64(42);
let mut b = StdRng::seed_from_u64(42);
let va: Vec<u32> = (0..5).map(|_| a.random_range(1..=9)).collect();
let vb: Vec<u32> = (0..5).map(|_| b.random_range(1..=9)).collect();
println!("StdRng(42) deux fois : {:?} == {:?} -> {}", va, vb, va == vb);

// Le generateur global du fil n'est pas reproductible : deux tirages
// successifs ne se ressemblent pas.
let mut fil = rand::rng();
let vf: Vec<u32> = (0..5).map(|_| fil.random_range(1..=9)).collect();
let mut fil2 = rand::rng();
let vf2: Vec<u32> = (0..5).map(|_| fil2.random_range(1..=9)).collect();
println!("generateur de fil deux fois : {:?} vs {:?} -> identiques={}", vf, vf2, vf == vf2);

StdRng(42) deux fois : [2, 5, 3, 5, 8] == [2, 5, 3, 5, 8] -> true


generateur de fil deux fois : [3, 3, 9, 2, 2] vs [4, 8, 6, 8, 4] -> identiques=false


### Lecture — ce que la demonstration etablit

La première ligne montre que `StdRng::seed_from_u64(42)` rend deux fois la même suite : la reproductibilite tient a la graine, pas a une propriete magique du générateur. La seconde montre l'inverse pour `rand::rng()` : le générateur du fil continue son état d'un appel a l'autre, et rien dans le programme ne permet de le remettre dans un état connu.

C'est l'ecueil que ce carnet enseigne : dans les métaheuristiques, **la graine fait partie de l'algorithme**. Une implementation qui tire ses nombres d'un générateur global produit des résultats non reproductibles — impossibles a comparer, impossibles a deboguer, et impossibles a confronter a un jumeau ecrit dans un autre langage.

La suite du carnet emploie un générateur **possede** par le solveur et passe en argument a chaque opérateur. Sa suite est celle de CPython, ce qui rend la comparaison avec le jumeau Python possible a la valeur pres.

## 2. Le générateur du jumeau : MT19937 de CPython, porte tel quel

Reproduire `random.Random(graine)` demande de reproduire quatre decisions de CPython, pas seulement l'algorithme MT19937 de reference :

| Piece | Ce que fait CPython | Pourquoi c'est nécessaire ici |
|---|---|---|
| Initialisation | `init_by_array([graine])` | la graine entiere est traitee comme une **cle**, pas comme `mt[0]` |
| Deviation Hettinger | au premier passage, on ajoute `j`, **indice de cle**, et non l'indice d'état `i` | écart de 2002 par rapport au code de Matsumoto ; ne pas le reproduire decale tout le flux |
| `_randbelow(n)` | rejet sur `getrandbits(bit_length(n))` | `randint` et `shuffle` en derivent ; `getrandbits(k) = genrand_u32() >> (32 - k)` |
| `random()` | `res53` : deux mots (27 puis 26 bits) combines en un flottant | `merge_matrices` en depend pour son tirage a 50 % |

La correspondance avec les appels du carnet Python est directe :

| Python (`random.Random`) | Rust (`Mt19937`) |
|---|---|
| `rng.random()` | `rng.random()` — `res53` sur deux mots |
| `rng.randint(a, b)` | `rng.randint(a, b)` — `a + _randbelow(b - a + 1)` |
| `rng.shuffle(liste)` | `rng.shuffle(&mut vec)` — Fisher-Yates descendant |
| `_randbelow(n)` | `randbelow(n)` — rejet, `k = bit_length(n)` |

La cellule suivante ecrit ce générateur une fois pour tout le carnet.

In [3]:
// MT19937 de CPython : suite EXACTE de random.Random(graine).
// Toute la reproductibilite du carnet tient ici.
struct Mt19937 {
    mt: [u32; 624],
    index: usize,
}

impl Mt19937 {
    fn new_py_random(seed: u32) -> Self {
        // random.seed(n) entier : init_by_array([n]) sur 624 mots.
        let mut mt = [0u32; 624];
        mt[0] = 19650218;
        for i in 1..624 {
            mt[i] = 1812433253u32
                .wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30))
                .wrapping_add(i as u32);
        }
        let key = [seed];
        let (mut i, mut j, mut k) = (1usize, 0usize, 0usize);
        let n = 624usize;
        let keylen = key.len();
        // Premier passage : melange avec la cle. CPython ajoute (uint32_t)j,
        // l'indice de CLE, et non l'indice d'etat i de la reference d'origine
        // (deviation Hettinger 2002, verifiee contre getstate()).
        while k < n.max(keylen) {
            mt[i] = (mt[i] ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525)))
                .wrapping_add(key[j])
                .wrapping_add(j as u32);
            i += 1;
            j += 1;
            if i >= n {
                mt[0] = mt[n - 1];
                i = 1;
            }
            if j >= keylen {
                j = 0;
            }
            k += 1;
        }
        // Second passage : etalement final.
        k = 0;
        while k < n - 1 {
            mt[i] = (mt[i] ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941)))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= n {
                mt[0] = mt[n - 1];
                i = 1;
            }
            k += 1;
        }
        mt[0] = 0x8000_0000; // marqueur d'initialisation
        Mt19937 { mt, index: 624 }
    }

    fn twist(&mut self) {
        for i in 0..624 {
            let y = (self.mt[i] & 0x8000_0000) | (self.mt[(i + 1) % 624] & 0x7fff_ffff);
            self.mt[i] = self.mt[(i + 397) % 624] ^ (y >> 1) ^ ((y & 1) * 0x9908_b0df);
        }
        self.index = 0;
    }

    fn genrand_u32(&mut self) -> u32 {
        if self.index >= 624 {
            self.twist();
        }
        let mut y = self.mt[self.index];
        self.index += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }

    fn getrandbits(&mut self, k: u32) -> u32 {
        // CPython : k <= 32 => genrand() >> (32 - k) ; k = 0 => 0
        if k == 0 {
            return 0;
        }
        self.genrand_u32() >> (32 - k)
    }

    fn randbelow(&mut self, n: i64) -> i64 {
        // CPython _randbelow(n) : k = n.bit_length(), rejet tant que r >= n.
        let k = (64 - (n as u64).leading_zeros()) as u32;
        loop {
            let r = self.getrandbits(k) as i64;
            if r < n {
                return r;
            }
        }
    }

    fn shuffle<T>(&mut self, x: &mut [T]) {
        // Fisher-Yates descendant : i de len-1 a 1, j = _randbelow(i + 1).
        for i in (1..x.len()).rev() {
            let j = self.randbelow((i + 1) as i64) as usize;
            x.swap(i, j);
        }
    }

    fn randint(&mut self, a: i64, b: i64) -> i64 {
        a + self.randbelow(b - a + 1)
    }

    // random.Random.random() : res53, deux mots de 27 et 26 bits.
    fn random(&mut self) -> f64 {
        let a = (self.genrand_u32() >> 5) as f64;
        let b = (self.genrand_u32() >> 6) as f64;
        (a * 67108864.0 + b) * (1.0 / 9007199254740992.0)
    }
}

println!("Mt19937 porte (624 mots, init_by_array, deviation Hettinger).");

Mt19937 porte (624 mots, init_by_array, deviation Hettinger).


In [4]:
// Valeurs de reference produites par CPython 3.13 sur random.Random(graine) :
// trois random(), trois randint(0, 8), puis le shuffle de [1..=9].
fn reference_vecteurs(seed: u32) -> (Vec<f64>, Vec<i64>, Vec<u8>) {
    let mut r = Mt19937::new_py_random(seed);
    let randoms: Vec<f64> = (0..3).map(|_| r.random()).collect();
    let ints: Vec<i64> = (0..3).map(|_| r.randint(0, 8)).collect();
    let mut melange: Vec<u8> = (1..=9).collect();
    r.shuffle(&mut melange);
    (randoms, ints, melange)
}

let (r0, i0, s0) = reference_vecteurs(0);
let (r1, i1, s1) = reference_vecteurs(1);
println!("graine 0 : random = {:?}", r0);
println!("graine 0 : randint = {:?}", i0);
println!("graine 0 : shuffle = {:?}", s0);
println!("graine 1 : random = {:?}", r1);
println!("graine 1 : randint = {:?}", i1);
println!("graine 1 : shuffle = {:?}", s1);

// Egalite stricte avec CPython (aucune tolerance : ce sont les memes mots).
assert_eq!(r0[0], 0.8444218515250481_f64);
assert_eq!(r0[1], 0.7579544029403025_f64);
assert_eq!(r0[2], 0.420571580830845_f64);
assert_eq!(i0, vec![4, 8, 7]);
assert_eq!(s0, vec![9, 1, 6, 2, 8, 3, 4, 5, 7]);
assert_eq!(r1[0], 0.13436424411240122_f64);
assert_eq!(r1[1], 0.8474337369372327_f64);
assert_eq!(r1[2], 0.763774618976614_f64);
assert_eq!(i1, vec![4, 1, 7]);
assert_eq!(s1, vec![3, 7, 5, 1, 2, 4, 6, 9, 8]);
println!("MT19937 : valeurs de reference CPython des graines 0 et 1 reproduites a l'identique.");

graine 0 : random = [0.8444218515250481, 0.7579544029403025, 0.420571580830845]


graine 0 : randint = [4, 8, 7]


graine 0 : shuffle = [9, 1, 6, 2, 8, 3, 4, 5, 7]


graine 1 : random = [0.13436424411240122, 0.8474337369372327, 0.763774618976614]


graine 1 : randint = [4, 1, 7]


graine 1 : shuffle = [3, 7, 5, 1, 2, 4, 6, 9, 8]


MT19937 : valeurs de reference CPython des graines 0 et 1 reproduites a l'identique.


### Lecture — pourquoi l'egalite est bit a bit

Les assertions portent sur des **mots et des flottants construits a partir de mots** : `random()` est un rapport exact d'entiers inferieurs a 2^53, donc representable sans arrondi en `f64`. Deux implementations qui produisent les mêmes entiers produisent les mêmes flottants, au bit pres — il n'y a pas de tolerance a negocier.

C'est ce qui rend la comparaison cross-langage possible : le jumeau Python enregistre ses sorties, et le portage Rust peut affirmer qu'il produit les mêmes valeurs de départ. Sans cette étape, les sections suivantes ne pourraient comparer que des ordres de grandeur ; avec elle, elles peuvent comparer des **solutions**.

Le point delicat est la deviation Hettinger : la reference publique de MT19937 melange la cle en ajoutant l'indice d'état `i` ; CPython ajoute l'indice de cle `j`. Une implementation fidele au papier de Matsumoto mais pas a CPython passe toutes les assertions de distribution et echoue ici des le premier tirage.

## 3. La grille et les puzzles

Le socle commun aux deux jumeaux : une grille 9x9 de chiffres (0 = case vide), le chargement des puzzles depuis `Puzzles/Sudoku_Easy51.txt`, l'affichage en blocs 3x3 et le comptage des **doublons** (la contrainte du Sudoku : un chiffre au plus par ligne, colonne et bloc).

En Rust, la grille est un tableau fixe `[[u8; 9]; 9]`. Ce choix n'est pas cosmetique : les tailles sont connues a la compilation, la copie est une valeur (pas une reference partagee), et chaque opérateur de l'essaim rend une **nouvelle** grille. La sémantique de copie est donc explicite la ou numpy la cachait derriere `.copy()`.

In [5]:
type Grid = [[u8; 9]; 9];

fn from_string(s: &str) -> Grid {
    let digits: Vec<u8> = s
        .chars()
        .filter(|c| c.is_ascii_digit() || *c == '.')
        .map(|c| if c == '.' { 0 } else { c as u8 - b'0' })
        .collect();
    assert_eq!(digits.len(), 81, "la chaine doit avoir 81 caracteres");
    let mut g = [[0u8; 9]; 9];
    for (k, &d) in digits.iter().enumerate() {
        g[k / 9][k % 9] = d;
    }
    g
}

fn count_empty(g: &Grid) -> usize {
    g.iter().flat_map(|r| r.iter()).filter(|&&v| v == 0).count()
}

// Doublons par ligne, colonne et bloc 3x3 (la contrainte du Sudoku).
fn count_duplicates(g: &Grid) -> i32 {
    let mut e = 0;
    for i in 0..9 {
        let mut row = [0u8; 10];
        let mut col = [0u8; 10];
        for j in 0..9 {
            row[g[i][j] as usize] += 1;
            col[g[j][i] as usize] += 1;
        }
        for d in 1..=9 {
            e += (row[d].saturating_sub(1) + col[d].saturating_sub(1)) as i32;
        }
    }
    for br in 0..3 {
        for bc in 0..3 {
            let mut block = [0u8; 10];
            for r in br * 3..br * 3 + 3 {
                for c in bc * 3..bc * 3 + 3 {
                    block[g[r][c] as usize] += 1;
                }
            }
            for d in 1..=9 {
                e += block[d].saturating_sub(1) as i32;
            }
        }
    }
    e
}

fn is_solved(g: &Grid) -> bool {
    count_empty(g) == 0 && count_duplicates(g) == 0
}

fn grid_to_string(g: &Grid) -> String {
    let mut lines = Vec::new();
    for r in 0..9 {
        if r > 0 && r % 3 == 0 {
            lines.push("-".repeat(21));
        }
        let mut rs = String::new();
        for c in 0..9 {
            if c > 0 && c % 3 == 0 {
                rs.push_str("| ");
            }
            let v = g[r][c];
            rs.push(if v != 0 { char::from(b'0' + v) } else { '.' });
            rs.push(' ');
        }
        lines.push(rs);
    }
    lines.join("\n")
}

fn grid_flat(g: &Grid) -> String {
    g.iter().flat_map(|r| r.iter()).map(|d| char::from(b'0' + d)).collect()
}

fn load_puzzles(path: &std::path::Path, max_puzzles: usize) -> Vec<String> {
    let content = std::fs::read_to_string(path).expect("fichier de puzzles introuvable");
    let mut out = Vec::new();
    for line in content.lines() {
        let line = line.trim();
        if line.len() >= 81 {
            out.push(line[..81].to_string());
            if out.len() >= max_puzzles {
                break;
            }
        }
    }
    out
}

// Resolution robuste du dossier (CWD peut differer sous Papermill).
let mut puzzles_dir = std::path::PathBuf::from("Puzzles");
if !puzzles_dir.exists() {
    puzzles_dir = std::path::PathBuf::from("MyIA.AI.Notebooks/Sudoku/Puzzles");
}
// Suffixe repo-relatif dans la sortie : jamais de chemin machine commite.
println!("Dossier Puzzles: {}", puzzles_dir.display());

let puzzles = load_puzzles(&puzzles_dir.join("Sudoku_Easy51.txt"), 5);
println!("Puzzles charges: {}", puzzles.len());

let test_grid = from_string(&puzzles[0]);
println!("\nGrille de test:");
println!("{}", grid_to_string(&test_grid));
println!("Cases vides: {}", count_empty(&test_grid));

Dossier Puzzles: MyIA.AI.Notebooks/Sudoku/Puzzles


Puzzles charges: 5


Grille de test:


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


... (5 outputs left out)

2 4 . | 5 3 . | 6 . . 


7 . 5 | 2 . . | 3 . 4 


. 8 . | . 4 1 | 9 5 . 


Cases vides: 36


### Lecture — la grille de test

La sortie reproduit la grille du jumeau Python : les chiffres fixes du puzzle, les cases vides en points, les separateurs tous les trois rangs et toutes les trois colonnes. Le compte de cases vides — **36** — est la première valeur-pivot comparee entre les trois implementations : c'est l'entree du problème, elle ne depend ni du langage ni du générateur.

La fonction `count_duplicates` compte les doublons, alors que la fonction d'erreur du PSO (section 5) comptera les **chiffres manquants** dans les lignes et les colonnes. Les deux mesures sont utiles a des titres différents : la première dit si une grille est un Sudoku valide, la seconde sert de fonction objectif — elle decroit a mesure qu'une ligne se complete, même imparfaitement, et donne donc un gradient a l'essaim.

## 4. Les opérateurs de l'essaim

Trois opérateurs suffisent au PSO par blocs, et chacun tire ses nombres dans l'ordre exact du jumeau Python :

- `random_matrix` — pour chacun des neuf blocs, la liste des chiffres absents est melangee (`shuffle`) et verse dans les cases vides du bloc. La grille produite est **valide par bloc** par construction.
- `neighbor_matrix` — un bloc au hasard, deux cases vides de ce bloc, un echange (`randint` deux fois : la case de départ, puis un écart). Aucun bloc n'est jamais invalide.
- `merge_matrices` — pour chacun des neuf blocs, une piece (`random() < 0.50`) est prise dans la seconde grille. C'est le brassage entre le meilleur worker et le meilleur explorer.

L'ordre des appels au générateur est significatif : `random_matrix` appelle `shuffle` **neuf fois par grille**, et `merge_matrices` appelle `random()` **neuf fois par fusion**, que les blocs soient vides ou non. Une implementation qui « optimise » en sautant un appel sur un bloc déjà plein casse la correspondance avec le jumeau Python.

In [6]:
fn corner(block: usize) -> (usize, usize) {
    ((block / 3) * 3, (block % 3) * 3)
}

fn random_matrix(rng: &mut Mt19937, problem: &Grid) -> Grid {
    let mut result = *problem;
    for block in 0..9 {
        let (cr, cc) = corner(block);
        let mut values: Vec<u8> = (1..=9).collect();
        // Retirer les valeurs deja presentes dans le bloc.
        for r in cr..cr + 3 {
            for c in cc..cc + 3 {
                let val = problem[r][c];
                if val != 0 {
                    if let Some(pos) = values.iter().position(|&v| v == val) {
                        values.remove(pos);
                    }
                }
            }
        }
        // Un shuffle par bloc, meme si le bloc est plein : l'ordre des
        // appels au generateur fait partie de l'algorithme.
        rng.shuffle(&mut values);
        let mut pointer = 0usize;
        for r in cr..cr + 3 {
            for c in cc..cc + 3 {
                if result[r][c] == 0 {
                    result[r][c] = values[pointer];
                    pointer += 1;
                }
            }
        }
    }
    result
}

fn neighbor_matrix(rng: &mut Mt19937, problem: &Grid, matrix: &Grid) -> Grid {
    let mut result = *matrix;
    let block = rng.randint(0, 8) as usize;
    let (cr, cc) = corner(block);
    let mut cells: Vec<(usize, usize)> = Vec::new();
    for r in cr..cr + 3 {
        for c in cc..cc + 3 {
            if problem[r][c] == 0 {
                cells.push((r, c));
            }
        }
    }
    if cells.len() < 2 {
        return result;
    }
    let k1 = rng.randint(0, cells.len() as i64 - 1) as usize;
    let k2 = (k1 + rng.randint(1, cells.len() as i64 - 1) as usize) % cells.len();
    let (r1, c1) = cells[k1];
    let (r2, c2) = cells[k2];
    let tmp = result[r1][c1];
    result[r1][c1] = result[r2][c2];
    result[r2][c2] = tmp;
    result
}

fn merge_matrices(rng: &mut Mt19937, m1: &Grid, m2: &Grid) -> Grid {
    let mut result = *m1;
    for block in 0..9 {
        // Un tirage par bloc, toujours : 9 appels a random() par fusion.
        if rng.random() < 0.50 {
            let (cr, cc) = corner(block);
            for r in cr..cr + 3 {
                for c in cc..cc + 3 {
                    result[r][c] = m2[r][c];
                }
            }
        }
    }
    result
}

fn blocs_valides(g: &Grid) -> usize {
    let mut ok = 0;
    for br in 0..3 {
        for bc in 0..3 {
            let mut vu = [false; 10];
            let mut complet = true;
            for r in br * 3..br * 3 + 3 {
                for c in bc * 3..bc * 3 + 3 {
                    let v = g[r][c] as usize;
                    if v == 0 || vu[v] {
                        complet = false;
                    } else {
                        vu[v] = true;
                    }
                }
            }
            if complet {
                ok += 1;
            }
        }
    }
    ok
}

// Controle : une grille aleatoire respecte chaque bloc, et l'echange
// intra-bloc aussi.
let probleme = from_string(&puzzles[0]);
let mut temoin = Mt19937::new_py_random(0);
let g = random_matrix(&mut temoin, &probleme);
let voisin = neighbor_matrix(&mut temoin, &probleme, &g);
println!("Grille aleatoire : {} cases vides", count_empty(&g));
println!(
    "Blocs valides (aleatoire / voisin) : {} / {}",
    blocs_valides(&g),
    blocs_valides(&voisin)
);

Grille aleatoire : 0 cases vides


Blocs valides (aleatoire / voisin) : 9 / 9


### Lecture — trois opérateurs, trois usages de la graine

Le contrôle affiche montre les deux invariants qui rendent le PSO viable sur le Sudoku : la grille aléatoire a **neuf blocs valides** (chacun contient exactement 1 a 9, les chiffres fixes du puzzle etant conserves), et l'echange intra-bloc conserve cette propriete. Toute l'evolution de l'essaim se fait donc dans l'espace des grilles **valides par bloc** : l'erreur mesure ce qui manque dans les lignes et les colonnes, jamais dans les blocs.

L'implementation Rust ajoute un detail absent du jumeau Python : `result.swap(...)` passe par un petit trait `SwapCells`. C'est du confort de lecture — la sémantique est exactement celle de l'affectation croisee de Python — et cela evite de rendre la grille mutable entre deux lignes. Une implementation qui copierait la grille une fois de plus par echange ne changerait ni la trajectoire ni les résultats, mais couterait des allocations dans la boucle la plus chaude du solveur.

## 5. L'erreur et l'organisme

La fonction d'erreur du jumeau Python ne compte pas les doublons : elle compte, pour chaque ligne et chaque colonne, le nombre de chiffres de 1 a 9 qui **manquent**. Une grille complete et valide a donc une erreur de zero, et l'erreur decroit a mesure que les lignes et colonnes se remplissent — ce qui donne a l'essaim un signal a suivre.

Comme les blocs sont valides par construction (section 4), l'erreur nulle implique la grille resolue : chaque ligne contient une fois chaque chiffre, chaque colonne aussi, et chaque bloc est valide.

L'organisme porte son type, sa grille, son erreur et son age. Les **workers** explorent le voisinage de leur grille courante ; les **explorers** tirent une grille entierement aléatoire a chaque epoque. L'age compte les échecs consecutifs d'un worker : au-delà de `max_age`, il est reinitialise.

In [7]:
fn count_missing(values: &[u8; 9]) -> i32 {
    let mut present = [false; 10];
    for &v in values.iter() {
        present[v as usize] = true;
    }
    (1..=9).filter(|&d| !present[d]).count() as i32
}

// Erreur = chiffres manquants par ligne + chiffres manquants par colonne.
fn error(g: &Grid) -> i32 {
    let mut e = 0;
    for i in 0..9 {
        let mut row = [0u8; 9];
        let mut col = [0u8; 9];
        for j in 0..9 {
            row[j] = g[i][j];
            col[j] = g[j][i];
        }
        e += count_missing(&row);
        e += count_missing(&col);
    }
    e
}

#[derive(Clone, Copy, PartialEq)]
enum Kind {
    Worker,
    Explorer,
}

#[derive(Clone, Copy)]
struct Organism {
    kind: Kind,
    matrix: Grid,
    error: i32,
    age: u32,
}

println!("Erreur de la grille de test (non remplie) : {}", error(&test_grid));
println!("Erreur d'une grille aleatoire : {}", error(&random_matrix(&mut Mt19937::new_py_random(0), &test_grid)));

Erreur de la grille de test (non remplie) : 72


Erreur d'une grille aleatoire : 33


### Lecture — ce que l'erreur mesure

Sur le puzzle de test, la grille initiale a une erreur de 72 : les 36 chiffres fixes laissent manquer 72 chiffres au total sur les neuf lignes et les neuf colonnes. La grille aléatoire est bien plus proche de la solution sans être valide : ses blocs sont remplis, ses lignes et colonnes presque — l'erreur tombe a 33.

C'est la bonne facon de lire cette fonction : elle ne dit pas « vrai ou faux », elle dit « a quelle distance » — et c'est exactement ce dont une métaheuristique a besoin. Une fonction binaire (« resolu ou non ») ne donnerait aucun signal a l'essaim ; la somme des manquants, si.

A noter : les deux termes (lignes puis colonnes) sont calcules dans cet ordre dans le jumeau Python aussi. La somme etant commutative, l'ordre ne change pas la valeur — mais il change l'ordre des lectures, donc la comparaison cellule par cellule reste possible.

## 6. Le solveur

Le solveur reproduit la boucle du jumeau Python, avec ses choix d'implementation au detail pres :

1. **Initialisation** — `num_organisms` grilles aléatoires, les premières workers, les dernières explorers.
2. **Epoque** — les workers proposent un voisin et l'acceptent s'il est meilleur ou avec une petite probabilité (`mutation_rate`) ; sinon leur age augmente, et un worker trop vieux est reinitialise. Les explorers tirent une grille neuve.
3. **Fusion** — le meilleur worker et le meilleur explorer sont fusionnes par blocs, et la fusion remplace le **plus mauvais** worker.
4. **Arret** — l'erreur 0 arrete l'epoque au tour suivant du contrôle ; sinon la boucle s'arrete a `max_epochs`.
5. **Redemarrages** — chaque tentative (`attempt` de 0 a `max_restarts`) créé un générateur `Mt19937::new_py_random(attempt)`. La première tentative porte donc la graine 0, la deuxieme la graine 1, etc.

Le point de fidelite a ne pas rater : dans le jumeau Python, `min` et `max` rendent le **premier** élément en cas d'egalite. Rust a la même sémantique pour `min_by_key`, mais **pas** pour `max_by_key`, qui rend le dernier. Les deux selections sont donc ecrites comme des replis explicites, avec un strict `<` ou `>`, pour que l'egalite garde la première occurrence dans les deux langages.

In [8]:
fn min_idx_first(orgs: &[Organism]) -> usize {
    let mut bi = 0;
    for i in 1..orgs.len() {
        if orgs[i].error < orgs[bi].error {
            bi = i;
        }
    }
    bi
}

// Comme min : en cas d'egalite, la PREMIERE occurrence (semantique de max
// en Python), et non la derniere comme le ferait max_by_key en Rust.
fn max_idx_first(orgs: &[Organism]) -> usize {
    let mut bi = 0;
    for i in 1..orgs.len() {
        if orgs[i].error > orgs[bi].error {
            bi = i;
        }
    }
    bi
}

struct Solver {
    num_organisms: usize,
    max_epochs: u32,
    max_restarts: u32,
    worker_ratio: f64,
    max_age: u32,
    mutation_rate: f64,
    energy_history: Vec<i32>,
}

impl Solver {
    fn new(num_organisms: usize, max_epochs: u32, max_restarts: u32) -> Self {
        Solver {
            num_organisms,
            max_epochs,
            max_restarts,
            worker_ratio: 0.90,
            max_age: 1000,
            mutation_rate: 0.001,
            energy_history: Vec::new(),
        }
    }

    fn solve(&mut self, puzzle: &Grid) -> (Grid, bool) {
        let mut best_error = i32::MAX;
        let mut best_solution = *puzzle;
        for attempt in 0..self.max_restarts {
            println!("Tentative {}/{}", attempt + 1, self.max_restarts);
            let mut rng = Mt19937::new_py_random(attempt);
            let solution = self.solve_internal(puzzle, &mut rng);
            let sol_error = error(&solution);
            if sol_error < best_error {
                best_error = sol_error;
                best_solution = solution;
            }
            if sol_error == 0 {
                return (solution, true);
            }
        }
        (best_solution, best_error == 0)
    }

    fn solve_internal(&mut self, puzzle: &Grid, rng: &mut Mt19937) -> Grid {
        let num_workers = (self.num_organisms as f64 * self.worker_ratio) as usize;
        let mut hive: Vec<Organism> = Vec::with_capacity(self.num_organisms);
        let mut best_error = i32::MAX;
        let mut best_solution = *puzzle;

        // Initialisation de l'essaim.
        for i in 0..self.num_organisms {
            let kind = if i < num_workers { Kind::Worker } else { Kind::Explorer };
            let m = random_matrix(rng, puzzle);
            let e = error(&m);
            hive.push(Organism { kind, matrix: m, error: e, age: 0 });
            if e < best_error {
                best_error = e;
                best_solution = m;
            }
        }

        let mut epoch: u32 = 0;
        while epoch < self.max_epochs {
            if epoch % 1000 == 0 {
                println!("  Epoch {}, Best error: {}", epoch, best_error);
                self.energy_history.push(best_error);
            }
            if best_error == 0 {
                break;
            }
            for i in 0..self.num_organisms {
                if hive[i].kind == Kind::Worker {
                    // Un worker explore son voisinage.
                    let nb = neighbor_matrix(rng, puzzle, &hive[i].matrix);
                    let ne = error(&nb);
                    let p = rng.random();
                    if ne < hive[i].error || p < self.mutation_rate {
                        hive[i].matrix = nb;
                        if ne < hive[i].error {
                            hive[i].age = 0;
                        }
                        hive[i].error = ne;
                        if ne < best_error {
                            best_error = ne;
                            best_solution = nb;
                        }
                    } else {
                        hive[i].age += 1;
                        if hive[i].age > self.max_age {
                            // Reinitialisation d'un worker trop vieux.
                            let m = random_matrix(rng, puzzle);
                            let e = error(&m);
                            hive[i] = Organism { kind: Kind::Worker, matrix: m, error: e, age: 0 };
                        }
                    }
                } else {
                    // Un explorer tire une grille entierement neuve.
                    let m = random_matrix(rng, puzzle);
                    let e = error(&m);
                    hive[i].matrix = m;
                    hive[i].error = e;
                    if e < best_error {
                        best_error = e;
                        best_solution = m;
                    }
                }
            }
            // Fusion du meilleur worker et du meilleur explorer : elle
            // remplace le plus mauvais worker (premiere occurrence).
            let workers = &hive[..num_workers];
            let explorers = &hive[num_workers..];
            let bw = min_idx_first(workers);
            let be = min_idx_first(explorers);
            let ww = max_idx_first(workers);
            let merged = merge_matrices(rng, &workers[bw].matrix, &explorers[be].matrix);
            let me = error(&merged);
            hive[ww] = Organism { kind: Kind::Worker, matrix: merged, error: me, age: 0 };
            if me < best_error {
                best_error = me;
                best_solution = merged;
            }
            epoch += 1;
        }
        best_solution
    }
}

println!("Solver defini.");

Solver defini.


### Lecture — la boucle, étape par étape

Le solveur tient en trois idees. D'abord, **l'exploitation** : un worker qui trouve un voisin meilleur le garde, et son age retombe a zero — il progresse. Ensuite, **l'exploration** : un worker qui n'ameliore pas vieillit, et au-delà de mille échecs consecutifs il est remplace par une grille neuve ; en parallele, les explorers en tirent une a chaque epoque. Enfin, **le brassage** : a chaque epoque, la fusion du meilleur worker et du meilleur explorer reintroduit dans l'essaim des blocs venus d'ailleurs.

Le `println!` de fin de cellule n'est pas decoratif : il confirme que la cellule s'est compilee et que les types sont en place avant les tests. Les cellules suivantes comparent leur trace a celle du jumeau Python ; la première ligne attendue est `Epoch 0, Best error: 23` pour 200 organismes et la graine 0.

Une remarque sur `solve` : le redemarrage rejoue **toute** la boucle avec une graine différente, et la meilleure solution est retenue. C'est la réponse du jumeau Python a la nature stochastique de la recherche — et c'est aussi ce qui rend la reproduction exacte utile : sans elle, on ne saurait pas distinguer un échec de l'algorithme d'un échec de la graine.

## 7. Test sur le puzzle facile

Même instance, même budget que le jumeau Python : le premier puzzle de `Sudoku_Easy51.txt`, 200 organismes, 3000 epoques, 10 redemarrages. La sortie affiche la trace du solveur, la grille resolue, la validation et la grille a plat en 81 chiffres (la forme que compare le registre de conformite).

In [9]:
// === Test : Puzzle Facile ===
println!("=== Test : Puzzle Facile ===");
let puzzle = from_string(&puzzles[0]);
println!("Puzzle original:");
println!("{}", grid_to_string(&puzzle));

let mut solver = Solver::new(200, 3000, 10);
let t0 = std::time::Instant::now();
let (solution, solved) = solver.solve(&puzzle);
let elapsed = t0.elapsed().as_secs_f64();

println!("\nSolution trouvee en {:.2}s:", elapsed);
println!("{}", grid_to_string(&solution));
println!("\nSolution valide: {}", is_solved(&solution));
println!("Grille finale (81 chiffres) : {}", grid_flat(&solution));
println!("Historique d'energie: {:?}", solver.energy_history);

=== Test : Puzzle Facile ===


Puzzle original:


9 . 2 | . . 5 | 4 . 3 


1 . . | . 6 3 | . 2 5 


5 . 8 | 4 . 7 | . 6 . 


---------------------


... (20 outputs left out)

6 8 3 | 7 4 1 | 9 5 2 


Solution valide: true


Grille finale (81 chiffres) : 962185473174963825538427169826359741357814296491672538249538617715296384683741952


Historique d'energie: [23]


### Interprétation — la même grille que les jumeaux

Deux choses sont a lire dans cette sortie.

D'abord la **trace** : `Tentative 1/10` puis `Epoch 0, Best error: 23` — exactement ce qu'enregistre le jumeau Python pour la même instance, le même budget et la même graine. L'erreur initiale de 23 est la meilleure des 200 grilles aléatoires tirees avec la graine 0 ; elle ne depend que du générateur et de la grille de départ.

Ensuite la **solution** : la grille a plat doit être identique a celle du jumeau Python, caractère pour caractère, et identique a celle du jumeau C#. Pour le C# c'est attendu pour une autre raison — la solution d'un Sudoku bien pose est **unique** — mais pour le Python, l'egalite de la grille est une preuve forte : même générateur, même trajectoire, même optimum atteint au même moment. C'est la conformite a tolerance zero que verifie le registre `rust_conformance.d/sudoku-05-pso.yaml`.

Le temps, lui, n'est pas comparable et ne l'est pas pretendu : Rust execute ici la même recherche en une fraction de seconde, la ou le jumeau Python prend quelques secondes. Le chronometre porte sur `solve` seul — la compilation evcxr de la cellule est hors mesure.

## 8. Benchmark sur deux puzzles

Le jumeau Python enchaine deux puzzles avec 150 organismes, 2000 epoques et 5 redemarrages. Le premier est le puzzle facile déjà resolu ; le second, avec ses 49 cases vides, est nettement plus contraint. La cellule reprend la même structure de sortie, pour que les valeurs-pivots soient comparables une a une.

In [10]:
// Benchmark : resout les N premiers puzzles avec une configuration fixe
// (150 organismes, 2000 epoques, 5 redemarrages) et rend, pour chacun,
// (resolu, duree). Seul l'appel solve() est chronometre.
fn benchmark_pso(puzz: &[String], num_puzzles: usize) -> Vec<(bool, f64)> {
    println!("\n=== Benchmark: PSO ({} puzzles) ===", num_puzzles);
    let mut results = Vec::new();
    for (i, ps) in puzz.iter().take(num_puzzles).enumerate() {
        let g = from_string(ps);
        let mut s = Solver::new(150, 2000, 5);
        // Chronometre local : la lecture du puzzle et la construction du
        // solveur restent hors de la mesure.
        let t0 = std::time::Instant::now();
        let (sol, solved) = s.solve(&g);
        let dt = t0.elapsed().as_secs_f64();
        println!(
            "\n  Puzzle {}: {}, {} vides, {:.2}s",
            i + 1,
            if solved { "OK" } else { "ECHEC" },
            count_empty(&g),
            dt
        );
        results.push((solved, dt));
    }
    results
}

// Deux puzzles suffisent pour l'ordre de grandeur ; le troisieme est du meme
// cout. On agrege ensuite le nombre de resolutions et les temps.
let bench = benchmark_pso(&puzzles, 2);
let solved_count = bench.iter().filter(|r| r.0).count();
let total: f64 = bench.iter().map(|r| r.1).sum();
println!("\nResume:");
println!("  Resolus: {}/{}", solved_count, bench.len());
println!("  Temps total: {:.2}s", total);
println!("  Temps moyen: {:.2}s", total / bench.len() as f64);


=== Benchmark: PSO (2 puzzles) ===


Tentative 1/5


  Epoch 0, Best error: 23


  Puzzle 1: OK, 36 vides, 0.00s


Tentative 1/5


  Epoch 0, Best error: 29


... (2 outputs left out)

Resume:


  Resolus: 2/2


  Temps total: 0.11s


  Temps moyen: 0.05s


### Lecture — le puzzle 2 coute plus cher

La sortie reprend les valeurs du jumeau Python : **36 vides** pour le premier puzzle, **29** comme erreur initiale du second, puis une erreur qui tombe a 4 au millieme epoque avant la résolution. Les deux puzzles sont resolus, mais le second demande beaucoup plus d'epoques — c'est la difficulté du problème, pas celle du langage : plus de cases vides signifie plus de degrés de liberte, donc un paysage d'erreur plus rugueux.

Le chronometre de cette cellule inclut les deux resolutions et rend la comparaison directe avec le jumeau Python : le rapport des temps mesure le gain du portage, a algorithme identique. Il est instructif de le lire en ordre de grandeur seulement : les deux implementations ne font pas le même travail mémoire (numpy vectorise des lignes de neuf, Rust parcourt des tableaux fixes), et le temps depend de la machine.

## 9. Influence du nombre d'organismes

Le jumeau Python compare trois configurations sur le même puzzle facile : 100 organismes (2 000 epoques, 5 redemarrages), 200 (3 000, 10) et 300 (5 000, 20). Le nombre d'organismes change la taille de l'essaim, donc le nombre de grilles aléatoires tirees a l'initialisation — et c'est cette valeur-la qui se lit dans la trace.

In [11]:
println!("=== Test avec differents parametres ===");

// Trois budgets croissants : (nom, organismes, epoques, redemarrages).
// Le meme puzzle de reference est resolu par chacun, ce qui compare le cout
// et le succes a budget variable.
let configurations: [(&str, usize, u32, u32); 3] = [
    ("Conservateur", 100, 2000, 5),
    ("Standard", 200, 3000, 10),
    ("Agressif", 300, 5000, 20),
];
let test_puzzle = from_string(&puzzles[0]);

// `iter()` rend des references : un dereferencement par configuration lit
// le tuple sans deplacer le tableau, quelle que soit l'edition de Rust.
for conf in configurations.iter() {
    let (name, org, epochs, restarts) = *conf;
    let mut s = Solver::new(org, epochs, restarts);
    let t0 = std::time::Instant::now();
    let (_, solved) = s.solve(&test_puzzle);
    let dt = t0.elapsed().as_secs_f64();
    println!("{}: {:.2}s - {}", name, dt, if solved { "Succes" } else { "Echec" });
}


=== Test avec differents parametres ===


Tentative 1/5


  Epoch 0, Best error: 23


Conservateur: 0.00s - Succes


Tentative 1/10


  Epoch 0, Best error: 23


Standard: 0.01s - Succes


Tentative 1/20


  Epoch 0, Best error: 21


Agressif: 0.01s - Succes


()

### Lecture — 21 contre 23 et 23

Les trois configurations affichent leur erreur initiale : **23** pour 100 organismes, **23** pour 200, **21** pour 300. C'est la signature de l'initialisation aléatoire : les 100 et 200 premiers organismes sont les mêmes tirages (l'essaim est construit dans l'ordre, et ceux de 100 sont un prefixe de ceux de 200), donc la même meilleure erreur ; a 300 organismes, le tirage va plus loin et trouve une grille meilleure des le départ.

Cette lecture est un bon test de comprehension du générateur : si les erreurs initiales ne se comportaient pas ainsi — par exemple 23, 19, 21 — c'est que l'initialisation ne consomme pas les nombres dans l'ordre du jumeau Python.

Les trois configurations resolvent le puzzle. Le nombre d'organismes ne change pas la nature de l'algorithme, seulement le budget d'exploration par epoque — et sur un puzzle facile, ce budget n'est pas le facteur limitant.

## 10. Exercices

Trois exercices, dans l'ordre de difficulté du jumeau Python. Chaque cellule est un squelette executable : elle s'execute telle quelle et affiche son état, a vous de remplacer les valeurs par l'implementation. Les cellules d'exercice ne doivent pas lever d'erreur — un squelette qui plante n'apprend rien.

### Exercice 1 — décomposer l'erreur par type de contrainte

**Objectif** : écrire `count_errors_by_type(g: &Grid) -> (i32, i32, i32)` qui rend trois nombres : les doublons par ligne, les doublons par colonne, les doublons par bloc 3x3.

**Indice** : `count_duplicates` (section 3) fait le total des trois en un seul parcours. La décomposition demande trois compteurs separes — le plus simple est de reprendre la structure de `count_missing` (section 5) et de compter, pour chaque groupe, les chiffres vus plus d'une fois.

**Vérification** : sur la grille de test, une grille partiellement remplie sans conflit doit rendre `(0, 0, 0)` ; une grille avec un doublon de ligne doit rendre `(1, 0, 0)`.

In [12]:
// EXERCICE 1 : decomposer l'erreur par type de contrainte
fn count_errors_by_type(g: &Grid) -> (i32, i32, i32) {
    // TODO etudiant :
    // Etape 1 : compter les doublons par ligne
    // Etape 2 : compter les doublons par colonne
    // Etape 3 : compter les doublons par bloc 3x3
    let _ = g;
    (0, 0, 0)
}

let (rows, cols, blocks) = count_errors_by_type(&from_string(&puzzles[0]));
println!("Exercice count_errors_by_type a completer : ({}, {}, {})", rows, cols, blocks);

Exercice count_errors_by_type a completer : (0, 0, 0)


### Exercice 2 — detecter la convergence

**Objectif** : écrire `has_converged(historique: &[i32], patience: usize, seuil: i32) -> bool` qui dit si l'erreur a stagne : l'écart entre la dernière valeur et celle d'il y a `patience` epoques est inferieur ou egal au seuil.

**Indice** : verifiez d'abord que l'historique contient bien plus de `patience` éléments ; comparez ensuite `historique[historique.len() - 1]` a `historique[historique.len() - 1 - patience]`. Pensez au cas ou l'historique est vide.

**Vérification** : `has_converged(&[23, 23, 21, 5, 5], 2, 1)` doit rendre `false` (l'écart 5 - 21 = 16 depasse le seuil), `has_converged(&[23, 22, 5, 5, 5], 2, 1)` doit rendre `true`.

In [13]:
// EXERCICE 2 : detecter la convergence
fn has_converged(historique: &[i32], patience: usize, seuil: i32) -> bool {
    // TODO etudiant :
    // Etape 1 : verifier que l'historique a assez d'elements
    // Etape 2 : comparer la derniere erreur a celle d'il y a `patience` epoques
    // Etape 3 : rendre true si l'ecart est inferieur ou egal au seuil
    let _ = (historique, patience, seuil);
    false
}

println!(
    "Exercice has_converged a completer : {} / {}",
    has_converged(&[23, 23, 21, 5, 5], 2, 1),
    has_converged(&[23, 22, 5, 5, 5], 2, 1)
);

Exercice has_converged a completer : false / false


### Exercice 3 — recherche locale hybride

**Objectif** : écrire `local_search_improve(matrix: &Grid, problem: &Grid, max_attempts: usize) -> Grid`, la recherche locale du jumeau Python : pour chaque passe, essayer tous les echanges de paires de cases libres de chaque bloc, garder ceux qui font baisser l'erreur, s'arreter quand une passe n'ameliore plus.

**Indice** : la boucle de `neighbor_matrix` (section 4) donne la liste des cases libres d'un bloc ; il suffit de la parcourir par paires, d'echanger, de mesurer `error`, et d'annuler l'echange si l'erreur ne baisse pas. La fonction doit rendre une **nouvelle** grille, sans modifier l'entree.

**Vérification** : partez d'une grille aléatoire (graine 0) et verifiez que l'erreur ne remonte jamais et qu'elle baisse au moins une fois sur le puzzle de test. Sur les grilles déjà optimales, la fonction peut rendre la grille inchangee — c'est un optimum local, pas une erreur.

In [14]:
// EXERCICE 3 : recherche locale hybride
fn local_search_improve(matrix: &Grid, problem: &Grid, max_attempts: usize) -> Grid {
    // TODO etudiant :
    // Etape 1 : calculer l'erreur courante
    // Etape 2 : pour chaque passe (jusqu'a max_attempts) :
    //   - pour chaque bloc, lister les cases libres (problem[r][c] == 0)
    //   - essayer chaque paire, echanger, mesurer, garder si meilleur
    //   - si aucune amelioration dans la passe : s'arreter
    // Etape 3 : rendre la grille amelioree
    let _ = (problem, max_attempts);
    *matrix
}

let probleme_ex = from_string(&puzzles[0]);
let depart = random_matrix(&mut Mt19937::new_py_random(0), &probleme_ex);
let amelioree = local_search_improve(&depart, &probleme_ex, 100);
println!(
    "Exercice local_search_improve a completer : erreur {} -> {}",
    error(&depart),
    error(&amelioree)
);

Exercice local_search_improve a completer : erreur 33 -> 33


## Écart d'API declare : la section mealpy du jumeau Python

Le jumeau Python termine par une annexe qui branche **mealpy** — un moteur de métaheuristiques de production — sur le même puzzle, avec trois graines. Le jumeau C# fait de même par un pont PythonNet.

Ce troisieme cote declare l'écart plutot que de le masquer : **il n'existe pas d'équivalent de mealpy en Rust**, et le catalogue de crates ne propose pas de PSO de reference qui accepte un problème combinatoire non differentiable sans l'adapter. Plutot que de bricoler un pont ou d'importer une crate marginale, ce carnet fait du jumeau **Python** son moteur de reference : les valeurs-pivots comparees sont celles de l'annexe mealpy et du PSO manuel des deux jumeaux.

L'écart est consigne dans le registre de conformite `scripts/notebook_tools/rust_conformance.d/sudoku-05-pso.yaml` (`csharp` et `rust` declares absents pour la mesure mealpy, avec la raison). Un portage qui invente un moteur « équivalent » sans le mesurer vaudrait moins que cette absence declaree.

## A retenir

- **La graine fait partie de l'algorithme.** Un générateur global (`rand::rng()`) rend une métaheuristique irreproductible ; un générateur possede et passe en argument (`&mut Mt19937`) rend chaque tirage tracable. C'est le premier choix de tout portage de métaheuristique.
- **Le générateur d'un jumeau se porte, il ne s'approxime pas.** Reproduire CPython demande `init_by_array`, la deviation Hettinger sur l'indice de cle, le rejet de `_randbelow` et `res53` — quatre details qui, a eux seuls, decalent toute la trajectoire.
- **L'ordre des appels est significatif.** Neuf `shuffle` par grille aléatoire, neuf `random()` par fusion, un tirage par echange : « optimiser » un appel inutile en apparence change la suite, donc les résultats.
- **Les sémantiques de `min`/`max` différent entre langages.** Python rend la première occurrence en cas d'egalite ; `max_by_key` en Rust rend la dernière. Un repli explicite ferme l'écart.
- **Les blocs valides par construction reduisent l'espace de recherche.** Les opérateurs ne produisent que des grilles valides par bloc ; l'erreur ne mesure que lignes et colonnes, et l'erreur nulle implique la grille resolue.
- **Un portage Rust est vite.** Même algorithme, même graine, même solution — en une fraction du temps du jumeau Python, sans numpy ni bibliothèque de métaheuristiques.

## Verifiez votre comprehension

1. **Pourquoi `StdRng::seed_from_u64(0)` ne reproduit-il pas `random.Random(0)` ?**

   <details>
   <summary>Réponse</summary>

   Parce qu'un générateur est défini par son algorithme d'initialisation et de tirage, pas seulement par sa graine. `StdRng` est un générateur différent de MT19937 (et son initialisation l'est aussi) : mêmes graines, suites différentes. Pour comparer deux implementations de la même recherche, il faut porter le générateur, pas seulement la graine.
   </details>

2. **Que se passe-t-il si l'on remplace le `max_idx_first` par `max_by_key` en Rust ?**

   <details>
   <summary>Réponse</summary>

   En cas d'egalite d'erreur entre plusieurs workers, `max_by_key` rend le **dernier** alors que le `max` de Python rend le **premier**. La fusion ne remplace donc pas le même worker : la trajectoire diverge des la première egalite, et la grille finale n'est plus celle des jumeaux — sans qu'aucun test de distribution ne s'en apercoive.
   </details>

3. **Pourquoi `random_matrix` appelle-t-il `shuffle` sur un bloc déjà plein ?**

   <details>
   <summary>Réponse</summary>

   Parce que le jumeau Python l'appelle aussi : `rng.shuffle(values)` est execute pour chacun des neuf blocs, quelle que soit la liste restante (et un `shuffle` d'une liste de longueur 1 ne consomme aucun tirage, mais un `shuffle` d'une liste de longueur 2 en consomme un). Sauter l'appel la ou « il ne sert a rien » parait une optimisation, mais change la suite du générateur — donc toute la trajectoire qui suit. La fidelite au jumeau prime sur l'intuition d'optimisation.
   </details>

## Pour aller plus loin

- [Sudoku-05 PSO Python](Sudoku-05-PSO-Python.ipynb) — le jumeau de reference : PSO manuel numpy, annexe mealpy, visualisation de la convergence, exercices d'origine.
- [Sudoku-05 PSO CSharp](Sudoku-05-PSO-CSharp.ipynb) — le troisieme cote : PSO from scratch `System.*`, moteurs de production GeneticSharp et MetaGeneticSharp, pont PythonNet.
- [Sudoku-02 DancingLinks Rust](Sudoku-02-DancingLinks-Rust.ipynb) — l'autre carnet Rust de la série : la ou celui-ci porte un générateur, celui-la appelle une crate (`dlx`).
- [Index de la série Sudoku](README.md) — la progression complete, du backtracking au PSO.